In [189]:
# import the libraries
import pandas as pd
import numpy as np

In [190]:
# importing df from github
url = 'https://raw.githubusercontent.com/MohammadErfanRashidi/Customer-Churn/refs/heads/main/data/Customer-Churn_Cleaned.csv'
df = pd.read_csv(url)

In [191]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [192]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [193]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [194]:
# check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

# remove duplicates if any exist
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Shape after removing duplicates: {df.shape}")

Duplicate rows found: 22
Shape after removing duplicates: (7021, 20)


In [195]:
# list of yes/no columns to convert
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

# convert yes/no to 1/0 and drop originals
for col in yes_no_cols:
    new_col_name = f'{col}_Num'
    df[new_col_name] = df[col].map({'Yes': 1, 'No': 0})
    df = df.drop(columns=[col])

print("Converted binary yes/no columns to numeric and dropped originals")
print(f"Added columns: {[f'{col}_Num' for col in yes_no_cols]}")

Converted binary yes/no columns to numeric and dropped originals
Added columns: ['Partner_Num', 'Dependents_Num', 'PhoneService_Num', 'PaperlessBilling_Num']


In [196]:
# convert gender to numeric and drop original
df['Gender_Num'] = df['gender'].map({'Male': 1, 'Female': 0})
df = df.drop(columns=['gender'])

print("Gender converted to Gender_Num (Male=1, Female=0)")

Gender converted to Gender_Num (Male=1, Female=0)


In [197]:
# convert target variable and drop original
df['Churn_Num'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['Churn'])

print("Churn converted to Churn_Num (Yes=1, No=0)")
print(f"Churn rate: {df['Churn_Num'].mean()*100:.2f}%")

Churn converted to Churn_Num (Yes=1, No=0)
Churn rate: 26.45%


In [198]:
# function to label encode using pandas factorize
def label_encode_pandas(df, column):
    new_col_name = f'{column}_Encoded'
    df[new_col_name] = pd.factorize(df[column])[0]
    print(f"{column} -> {new_col_name} ({df[new_col_name].nunique()} categories)")
    return new_col_name

# columns to encode
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
                  'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                  'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

encoded_cols = []
for col in multi_cat_cols:
    new_col = label_encode_pandas(df, col)
    encoded_cols.append(new_col)

# drop original columns
df = df.drop(columns=multi_cat_cols)

print(f"\nEncoded {len(encoded_cols)} columns and dropped originals")

MultipleLines -> MultipleLines_Encoded (3 categories)
InternetService -> InternetService_Encoded (3 categories)
OnlineSecurity -> OnlineSecurity_Encoded (3 categories)
OnlineBackup -> OnlineBackup_Encoded (3 categories)
DeviceProtection -> DeviceProtection_Encoded (3 categories)
TechSupport -> TechSupport_Encoded (3 categories)
StreamingTV -> StreamingTV_Encoded (3 categories)
StreamingMovies -> StreamingMovies_Encoded (3 categories)
Contract -> Contract_Encoded (3 categories)
PaymentMethod -> PaymentMethod_Encoded (4 categories)

Encoded 10 columns and dropped originals


In [199]:
# show mapping for important columns to understand the encoding
print("=== Encoding Mappings (for reference) ===")
print("\nContract_Encoded:")
print("0 = Month-to-month, 1 = One year, 2 = Two year")

print("\nInternetService_Encoded:")
print("0 = DSL, 1 = Fiber optic, 2 = No")

print("\nPaymentMethod_Encoded:")
print("0 = Electronic check, 1 = Mailed check, 2 = Bank transfer, 3 = Credit card")

=== Encoding Mappings (for reference) ===

Contract_Encoded:
0 = Month-to-month, 1 = One year, 2 = Two year

InternetService_Encoded:
0 = DSL, 1 = Fiber optic, 2 = No

PaymentMethod_Encoded:
0 = Electronic check, 1 = Mailed check, 2 = Bank transfer, 3 = Credit card


In [200]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Partner_Num,Dependents_Num,PhoneService_Num,PaperlessBilling_Num,Gender_Num,Churn_Num,MultipleLines_Encoded,InternetService_Encoded,OnlineSecurity_Encoded,OnlineBackup_Encoded,DeviceProtection_Encoded,TechSupport_Encoded,StreamingTV_Encoded,StreamingMovies_Encoded,Contract_Encoded,PaymentMethod_Encoded
0,0,1,29.85,29.85,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,0,34,56.95,1889.50,0,0,1,0,1,0,1,0,1,1,1,0,0,0,1,1
2,0,2,53.85,108.15,0,0,1,1,1,1,1,0,1,0,0,0,0,0,0,1
3,0,45,42.30,1840.75,0,0,0,0,1,0,0,0,1,1,1,1,0,0,1,2
4,0,2,70.70,151.65,0,0,1,1,0,1,1,1,0,1,0,0,0,0,0,0


In [201]:
# check data types after conversion
print("=== Data Types After Conversion ===")
print(df.dtypes)

# verify no object columns remain
object_cols = df.select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"\nWarning: Object columns still present: {object_cols}")
else:
    print("\n✓ All columns are now numeric")

=== Data Types After Conversion ===
SeniorCitizen                 int64
tenure                        int64
MonthlyCharges              float64
TotalCharges                float64
Partner_Num                   int64
Dependents_Num                int64
PhoneService_Num              int64
PaperlessBilling_Num          int64
Gender_Num                    int64
Churn_Num                     int64
MultipleLines_Encoded         int64
InternetService_Encoded       int64
OnlineSecurity_Encoded        int64
OnlineBackup_Encoded          int64
DeviceProtection_Encoded      int64
TechSupport_Encoded           int64
StreamingTV_Encoded           int64
StreamingMovies_Encoded       int64
Contract_Encoded              int64
PaymentMethod_Encoded         int64
dtype: object

✓ All columns are now numeric


In [202]:
# create tenure groups as numeric categories
# 0: 0-12 months (high churn)
# 1: 13-24 months (medium churn)
# 2: 25-48 months (low churn)
# 3: 49+ months (very low churn)

df['TenureGroup'] = pd.cut(df['tenure'], 
                           bins=[-1, 12, 24, 48, 100], 
                           labels=[0, 1, 2, 3])
df['TenureGroup'] = df['TenureGroup'].astype(int)

print("TenureGroup created (numeric):")
for i in range(4):
    months = ['0-12', '13-24', '25-48', '49+'][i]
    count = (df['TenureGroup'] == i).sum()
    churn_rate = df[df['TenureGroup'] == i]['Churn_Num'].mean() * 100
    print(f"  {i} ({months}): {count} customers, churn rate: {churn_rate:.1f}%")

TenureGroup created (numeric):
  0 (0-12): 2164 customers, churn rate: 47.4%
  1 (13-24): 1024 customers, churn rate: 28.7%
  2 (25-48): 1594 customers, churn rate: 20.4%
  3 (49+): 2239 customers, churn rate: 9.5%


In [203]:
# create charge groups as numeric categories
# 0: <$35 (low churn)
# 1: $35-70 (medium churn)
# 2: $70-100 (high churn)
# 3: $100+ (very high churn)

df['ChargeGroup'] = pd.cut(df['MonthlyCharges'], 
                           bins=[0, 35, 70, 100, 150], 
                           labels=[0, 1, 2, 3])
df['ChargeGroup'] = df['ChargeGroup'].astype(int)

print("ChargeGroup created (numeric):")
for i in range(4):
    ranges = ['<$35', '$35-70', '$70-100', '$100+'][i]
    count = (df['ChargeGroup'] == i).sum()
    churn_rate = df[df['ChargeGroup'] == i]['Churn_Num'].mean() * 100
    print(f"  {i} ({ranges}): {count} customers, churn rate: {churn_rate:.1f}%")

ChargeGroup created (numeric):
  0 (<$35): 1721 customers, churn rate: 10.7%
  1 ($35-70): 1719 customers, churn rate: 23.7%
  2 ($70-100): 2679 customers, churn rate: 37.8%
  3 ($100+): 902 customers, churn rate: 28.0%


In [204]:
# calculate average monthly spending
# tenure+1 avoids division by zero
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

print("AvgMonthlySpend statistics:")
print(df['AvgMonthlySpend'].describe().round(2))

AvgMonthlySpend statistics:
count    7021.00
mean       59.12
std        30.54
min         0.00
25%        26.27
50%        61.15
75%        84.89
max       118.97
Name: AvgMonthlySpend, dtype: float64


In [205]:
# count how many additional services each customer has
service_cols = ['OnlineSecurity_Encoded', 'OnlineBackup_Encoded', 
                'DeviceProtection_Encoded', 'TechSupport_Encoded', 
                'StreamingTV_Encoded', 'StreamingMovies_Encoded']

# for these encoded columns, check if 0 or 1 means 'Yes'
# since factorize assigns 0 to first category alphabetically
# 'No' comes before 'Yes' so 0='No', 1='Yes'
df['ServiceCount'] = df[service_cols].sum(axis=1)

print("ServiceCount created (number of additional services):")
print(df['ServiceCount'].value_counts().sort_index())

ServiceCount created (number of additional services):
ServiceCount
0      209
1     1038
2     1340
3     1240
4      975
5      576
6      131
12    1512
Name: count, dtype: int64


In [206]:
# create binary indicators from encoded columns

# has multiple lines (MultipleLines: 0='No', 1='No phone service', 2='Yes')
# wait, need to check actual encoding
# let's check the mapping
print("MultipleLines encoding check:")
print(df[['MultipleLines_Encoded']].value_counts().sort_index())

# assuming 2='Yes' after checking
df['HasMultipleLines'] = (df['MultipleLines_Encoded'] == 2).astype(int)

# internet service indicators (0='DSL', 1='Fiber optic', 2='No')
df['HasInternet'] = (df['InternetService_Encoded'] != 2).astype(int)
df['HasFiberOptic'] = (df['InternetService_Encoded'] == 1).astype(int)
df['HasDSL'] = (df['InternetService_Encoded'] == 0).astype(int)

# security services (0='No', 1='No internet service', 2='Yes')
df['HasOnlineSecurity'] = (df['OnlineSecurity_Encoded'] == 2).astype(int)
df['HasTechSupport'] = (df['TechSupport_Encoded'] == 2).astype(int)

print("\nBinary features created:")
print(f"  HasMultipleLines: {df['HasMultipleLines'].sum()}")
print(f"  HasInternet: {df['HasInternet'].sum()}")
print(f"  HasFiberOptic: {df['HasFiberOptic'].sum()}")
print(f"  HasDSL: {df['HasDSL'].sum()}")
print(f"  HasOnlineSecurity: {df['HasOnlineSecurity'].sum()}")
print(f"  HasTechSupport: {df['HasTechSupport'].sum()}")

MultipleLines encoding check:
MultipleLines_Encoded
0                         682
1                        3368
2                        2971
Name: count, dtype: int64

Binary features created:
  HasMultipleLines: 2971
  HasInternet: 5509
  HasFiberOptic: 3090
  HasDSL: 2419
  HasOnlineSecurity: 1512
  HasTechSupport: 1512


In [207]:
# sum of all services (internet + phone + additional services)
df['TotalServices'] = df['HasInternet'] + df['PhoneService_Num'] + df['ServiceCount']

print("TotalServices distribution:")
print(df['TotalServices'].value_counts().sort_index())

TotalServices distribution:
TotalServices
1       31
2      305
3     1071
4     1333
5     1213
6      914
7      531
8      111
13    1512
Name: count, dtype: int64


In [208]:
# customers with partner OR dependents (more stable)
df['HasPartnerOrDependents'] = (
    (df['Partner_Num'] == 1) | (df['Dependents_Num'] == 1)
).astype(int)

print(f"HasPartnerOrDependents: {df['HasPartnerOrDependents'].sum()} customers")

HasPartnerOrDependents: 3763 customers


In [209]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Partner_Num,Dependents_Num,PhoneService_Num,PaperlessBilling_Num,Gender_Num,Churn_Num,...,AvgMonthlySpend,ServiceCount,HasMultipleLines,HasInternet,HasFiberOptic,HasDSL,HasOnlineSecurity,HasTechSupport,TotalServices,HasPartnerOrDependents
0,0,1,29.85,29.85,1,0,0,1,0,0,...,14.925000,0,0,1,0,1,0,0,1,1
1,0,34,56.95,1889.50,0,0,1,0,1,0,...,53.985714,3,0,1,0,1,0,0,5,0
2,0,2,53.85,108.15,0,0,1,1,1,1,...,36.050000,1,0,1,0,1,0,0,3,0
3,0,45,42.30,1840.75,0,0,0,0,1,0,...,40.016304,4,0,1,0,1,0,0,5,0
4,0,2,70.70,151.65,0,0,1,1,0,1,...,50.550000,1,0,1,1,0,0,0,3,0


In [210]:
df.info()

<class 'pandas.DataFrame'>
Index: 7021 entries, 0 to 7042
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   SeniorCitizen             7021 non-null   int64  
 1   tenure                    7021 non-null   int64  
 2   MonthlyCharges            7021 non-null   float64
 3   TotalCharges              7021 non-null   float64
 4   Partner_Num               7021 non-null   int64  
 5   Dependents_Num            7021 non-null   int64  
 6   PhoneService_Num          7021 non-null   int64  
 7   PaperlessBilling_Num      7021 non-null   int64  
 8   Gender_Num                7021 non-null   int64  
 9   Churn_Num                 7021 non-null   int64  
 10  MultipleLines_Encoded     7021 non-null   int64  
 11  InternetService_Encoded   7021 non-null   int64  
 12  OnlineSecurity_Encoded    7021 non-null   int64  
 13  OnlineBackup_Encoded      7021 non-null   int64  
 14  DeviceProtection_Encoded

In [211]:
# manual standard scaling: (x - mean) / std
def manual_standard_scale(series):
    return (series - series.mean()) / series.std()

# features to scale
scale_cols = ['tenure', 'MonthlyCharges', 'ServiceCount', 'AvgMonthlySpend', 'TotalCharges']

scaled_cols = []
for col in scale_cols:
    new_col_name = f'{col}_Scaled'
    df[new_col_name] = manual_standard_scale(df[col])
    scaled_cols.append(new_col_name)
    print(f"{col}_Scaled: mean={df[new_col_name].mean():.2f}, std={df[new_col_name].std():.2f}")

print(f"\nCreated {len(scaled_cols)} scaled features")

tenure_Scaled: mean=0.00, std=1.00
MonthlyCharges_Scaled: mean=-0.00, std=1.00
ServiceCount_Scaled: mean=0.00, std=1.00
AvgMonthlySpend_Scaled: mean=-0.00, std=1.00
TotalCharges_Scaled: mean=-0.00, std=1.00

Created 5 scaled features


In [212]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Partner_Num,Dependents_Num,PhoneService_Num,PaperlessBilling_Num,Gender_Num,Churn_Num,...,HasDSL,HasOnlineSecurity,HasTechSupport,TotalServices,HasPartnerOrDependents,tenure_Scaled,MonthlyCharges_Scaled,ServiceCount_Scaled,AvgMonthlySpend_Scaled,TotalCharges_Scaled
0,0,1,29.85,29.85,1,0,0,1,0,0,...,1,0,0,1,1,-1.282637,-1.164052,-1.174157,-1.447328,-0.995615
1,0,34,56.95,1889.50,0,0,1,0,1,0,...,1,0,0,5,0,0.062382,-0.262792,-0.428071,-0.168120,-0.175249
2,0,2,53.85,108.15,0,0,1,1,1,1,...,1,0,0,3,0,-1.241879,-0.365888,-0.925461,-0.755501,-0.961074
3,0,45,42.30,1840.75,0,0,0,0,1,0,...,1,0,0,5,0,0.510722,-0.750005,-0.179375,-0.625607,-0.196755
4,0,2,70.70,151.65,0,0,1,1,0,1,...,0,0,0,3,0,-1.241879,0.194490,-0.925461,-0.280637,-0.941884


In [213]:
# drop intermediate capped columns since we have scaled versions
cols_to_drop = ['tenure', 'MonthlyCharges', 'ServiceCount', 'AvgMonthlySpend', 'TotalCharges']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print(f"Dropped intermediate columns")
print(f"Current shape: {df.shape}")

Dropped intermediate columns
Current shape: (7021, 32)


In [214]:
df.shape

(7021, 32)

In [215]:
df.head()

,SeniorCitizen,Partner_Num,Dependents_Num,PhoneService_Num,PaperlessBilling_Num,Gender_Num,Churn_Num,MultipleLines_Encoded,InternetService_Encoded,OnlineSecurity_Encoded,...,HasDSL,HasOnlineSecurity,HasTechSupport,TotalServices,HasPartnerOrDependents,tenure_Scaled,MonthlyCharges_Scaled,ServiceCount_Scaled,AvgMonthlySpend_Scaled,TotalCharges_Scaled
0,0,1,0,0,1,0,0,0,0,0,...,1,0,0,1,1,-1.282637,-1.164052,-1.174157,-1.447328,-0.995615
1,0,0,0,1,0,1,0,1,0,1,...,1,0,0,5,0,0.062382,-0.262792,-0.428071,-0.168120,-0.175249
2,0,0,0,1,1,1,1,1,0,1,...,1,0,0,3,0,-1.241879,-0.365888,-0.925461,-0.755501,-0.961074
3,0,0,0,0,0,1,0,0,0,1,...,1,0,0,5,0,0.510722,-0.750005,-0.179375,-0.625607,-0.196755
4,0,0,0,1,1,0,1,1,1,0,...,0,0,0,3,0,-1.241879,0.194490,-0.925461,-0.280637,-0.941884


In [216]:
# export the processed dataset
df.to_csv('Customer-Churn_Processed.csv', index = False)